# Food Supply Chain Sales & Profitability Analysis

**Business question:** Which product categories and regions drive profitability, and where should a food manufacturer focus to improve margins and reduce supply chain costs?

This notebook loads the Superstore dataset, maps retail categories to a food manufacturer's portfolio (Beverages, Snacks, Perishables), cleans the data, and produces the KPIs, charts and dashboard data files.

In [ ]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
OUT_CHARTS = ROOT / "outputs" / "charts"
OUT_DOCS = ROOT / "docs" / "data"
OUT_CHARTS.mkdir(parents=True, exist_ok=True)
OUT_DOCS.mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)

## 1. Load the data
The raw file is `data/superstore.csv`. Dates are parsed, and the retail `Product Category` is mapped to a food portfolio so the analysis reads like a food manufacturer's distribution network.

In [ ]:
df = pd.read_csv(DATA / "superstore.csv")
print("Raw shape:", df.shape)
df.head()

In [ ]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

CATEGORY_MAP = {
    "Technology": "Beverages",
    "Furniture": "Snacks",
    "Office Supplies": "Perishables",
}
df["Food Category"] = df["Product Category"].map(CATEGORY_MAP).fillna("Other")

## 2. Clean the data
- Drop rows without sales, profit or order date.
- Coerce numeric columns.
- Remove duplicate order lines.
- Clip discounts to a sane 0–1 range.

In [ ]:
numeric_cols = ["Order Quantity", "Sales", "Discount", "Profit", "Unit Price",
                "Shipping Cost", "Product Base Margin"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=["Sales", "Profit", "Order Date"])
df = df.drop_duplicates(subset=["Row ID"])
df["Discount"] = df["Discount"].clip(lower=0, upper=1)
df["Shipping Cost"] = df["Shipping Cost"].fillna(0)
df["Year"] = df["Order Date"].dt.year
print(f"Dropped {before - len(df)} rows during cleaning. Clean shape: {df.shape}")

## 3. KPI overview
Headline numbers for the dashboard: total sales, profit margin, average discount, and year-over-year sales growth.

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = total_profit / total_sales * 100
avg_discount = df["Discount"].mean() * 100

yearly = df.groupby("Year")["Sales"].sum().sort_index()
if len(yearly) >= 2:
    sales_growth = (yearly.iloc[-1] / yearly.iloc[-2] - 1) * 100
else:
    sales_growth = 0.0

print(f"Total Sales:      ${total_sales:,.2f}")
print(f"Total Profit:     ${total_profit:,.2f}")
print(f"Profit Margin:    {profit_margin:.2f}%")
print(f"Avg Discount:     {avg_discount:.2f}%")
print(f"Sales Growth YoY: {sales_growth:.2f}%")

## 4. Category profitability
Which food categories drive sales and profit? This answers the first half of the business question.

In [ ]:
cat = (df.groupby("Food Category")
         .agg(total_sales=("Sales", "sum"),
              total_profit=("Profit", "sum"),
              avg_discount=("Discount", "mean"),
              order_lines=("Row ID", "count"))
         .reset_index())
cat["profit_margin_pct"] = cat["total_profit"] / cat["total_sales"] * 100
cat = cat.sort_values("total_sales", ascending=False)
cat

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(cat))
ax.bar(x - 0.2, cat["total_sales"], width=0.4, label="Sales", color="#2E86AB")
ax.bar(x + 0.2, cat["total_profit"], width=0.4, label="Profit", color="#F18F01")
ax.set_xticks(x)
ax.set_xticklabels(cat["Food Category"])
ax.set_ylabel("CAD")
ax.set_title("Sales vs Profit by Food Category")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "category_sales_profit.png", dpi=120)
plt.close(fig)

## 5. Regional profitability
Profit per province shows where the distribution network is strongest and weakest.

In [ ]:
region = (df.groupby("Province")
            .agg(total_sales=("Sales", "sum"),
                 total_profit=("Profit", "sum"),
                 avg_discount=("Discount", "mean"),
                 shipping_cost=("Shipping Cost", "sum"))
            .reset_index())
region["profit_margin_pct"] = region["total_profit"] / region["total_sales"] * 100
region = region.sort_values("total_profit", ascending=False)
region.head(10)

In [ ]:
top_regions = region.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_regions["Province"], top_regions["total_profit"], color="#3BB273")
ax.set_xlabel("Total Profit (CAD)")
ax.set_title("Top 10 Provinces by Profit")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "region_profit.png", dpi=120)
plt.close(fig)

## 6. Discount impact
Discounting is the classic margin killer. Bucketing orders by discount band shows the trade-off between volume and margin.

In [ ]:
def discount_band(d):
    if d == 0:
        return "0% (No discount)"
    if d <= 0.05:
        return "1-5%"
    if d <= 0.10:
        return "6-10%"
    if d <= 0.20:
        return "11-20%"
    return "20%+"

df["Discount Band"] = df["Discount"].apply(discount_band)
disc = (df.groupby("Discount Band")
          .agg(total_sales=("Sales", "sum"),
               total_profit=("Profit", "sum"),
               order_lines=("Row ID", "count"),
               avg_profit=("Profit", "mean"))
          .reset_index())
disc["profit_margin_pct"] = disc["total_profit"] / disc["total_sales"] * 100
band_order = ["0% (No discount)", "1-5%", "6-10%", "11-20%", "20%+"]
disc["sort"] = disc["Discount Band"].map({b: i for i, b in enumerate(band_order)})
disc = disc.sort_values("sort").drop(columns="sort")
disc

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(disc["Discount Band"], disc["profit_margin_pct"], marker="o", color="#C73E1D", linewidth=2)
ax.set_ylabel("Profit Margin (%)")
ax.set_title("Profit Margin by Discount Band")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_CHARTS / "discount_margin.png", dpi=120)
plt.close(fig)

## 7. Shipping mode cost efficiency
Shipping cost as a share of sales reveals which modes erode margin.

In [ ]:
ship = (df.groupby("Ship Mode")
          .agg(total_sales=("Sales", "sum"),
               shipping_cost=("Shipping Cost", "sum"),
               avg_profit=("Profit", "mean"),
               order_lines=("Row ID", "count"))
          .reset_index())
ship["shipping_cost_pct"] = ship["shipping_cost"] / ship["total_sales"] * 100
ship = ship.sort_values("shipping_cost_pct", ascending=False)
ship

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(ship["Ship Mode"], ship["shipping_cost_pct"], color="#6A4C93")
ax.set_ylabel("Shipping Cost (% of Sales)")
ax.set_title("Shipping Cost as % of Sales by Ship Mode")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "ship_mode_cost.png", dpi=120)
plt.close(fig)

## 8. Top products by profit
The top 5 products by profit, plus a count of loss-making products to flag for review.

In [ ]:
prod = (df.groupby(["Product Name", "Food Category"])
          .agg(total_sales=("Sales", "sum"),
               total_profit=("Profit", "sum"))
          .reset_index())
prod["profit_margin_pct"] = prod["total_profit"] / prod["total_sales"] * 100
top_products = prod.sort_values("total_profit", ascending=False).head(5).reset_index(drop=True)
loss_making = int((prod["total_profit"] < 0).sum())
top_products

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
short_names = [n[:30] + "..." if len(n) > 30 else n for n in top_products["Product Name"]]
ax.barh(short_names[::-1], top_products["total_profit"][::-1], color="#F18F01")
ax.set_xlabel("Total Profit (CAD)")
ax.set_title("Top 5 Products by Profit")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "top_products.png", dpi=120)
plt.close(fig)

## 9. Save summary and dashboard data
The flat `summary.json` feeds the KPI cards; each `docs/data/*.json` file feeds a dashboard chart.

In [ ]:
summary = {
    "total_sales": round(float(total_sales), 2),
    "total_profit": round(float(total_profit), 2),
    "profit_margin_pct": round(float(profit_margin), 2),
    "avg_discount_pct": round(float(avg_discount), 2),
    "sales_growth_yoy_pct": round(float(sales_growth), 2),
    "order_lines": int(len(df)),
    "loss_making_products": loss_making,
    "top_category": str(cat.iloc[0]["Food Category"]),
    "top_province": str(region.iloc[0]["Province"]),
}
with open(ROOT / "outputs" / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
cat_out = cat[["Food Category", "total_sales", "total_profit", "profit_margin_pct"]].copy()
cat_out.columns = ["category", "total_sales", "total_profit", "profit_margin_pct"]
cat_out.to_json(OUT_DOCS / "category_profit.json", orient="records")

region_out = region[["Province", "total_profit", "profit_margin_pct"]].head(10).copy()
region_out.columns = ["province", "total_profit", "profit_margin_pct"]
region_out.to_json(OUT_DOCS / "region_profit.json", orient="records")

disc_out = disc[["Discount Band", "profit_margin_pct", "total_profit"]].copy()
disc_out.columns = ["discount_band", "profit_margin_pct", "total_profit"]
disc_out.to_json(OUT_DOCS / "discount_margin.json", orient="records")

ship_out = ship[["Ship Mode", "shipping_cost_pct"]].copy()
ship_out.columns = ["ship_mode", "shipping_cost_pct"]
ship_out.to_json(OUT_DOCS / "ship_mode.json", orient="records")

top_out = top_products[["Product Name", "total_profit"]].copy()
top_out.columns = ["product_name", "total_profit"]
top_out.to_json(OUT_DOCS / "top_products.json", orient="records")

print("Dashboard data files written to", OUT_DOCS)
print("Charts written to", OUT_CHARTS)